In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/00010
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  690.3960803722349
RUN  2 , total integrated cost =  106.07784409254252
RUN  3 , total integrated cost =  74.87618232783505
RUN  4 , total integrated cost =  60.90625076394285
RUN  5 , total integrated cost =  52.28919269543209
RUN  6 , total integrated cost =  36.97302732287212
RUN  7 , total integrated cost =  36.523328159507564
RUN  8 , total integrated cost =  28.856199069220793
RUN  9 , total integrated cost =  28.787235852743226
RUN  10 , total integrated cost =  28.748509092470652
RUN  11 , total integrated cost =  28.70663626990607
RUN  12 , total integrated cost =  28.680045992115275
RUN  13 , total integrated cost =  28.647474143316774
RUN  14 , total integrated cost =  28.623986756540102
RUN  15 , total integrated cost =  28.588817698574477
RUN

ERROR:root:Problem in initial value trasfer


RUN  90 , total integrated cost =  25.9893678104781
Control only changes marginally.
RUN  91 , total integrated cost =  25.9893678104781
Improved over  91  iterations in  13.160501822829247  seconds by  99.49013360655505  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446976024539 -56.624469634936794
weight =  1961.2981221284863
set cost params:  1.0 0.0 1961.2981221284863
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.929065321389
Gradient descend method:  None
RUN  1 , total integrated cost =  4930.459602213505
RUN  2 , total integrated cost =  4930.459602213496
RUN  3 , total integrated cost =  4930.459602213494


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  4930.459602213494
Control only changes marginally.
RUN  4 , total integrated cost =  4930.459602213494
Improved over  4  iterations in  0.18711398355662823  seconds by  3.190098684353913  percent.
Problem in initial value trasfer:  Vmean_exc -56.62659014344855 -56.62656380958566
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  605.0651175557289
RUN  2 , total integrated cost =  133.59238071760223
RUN  3 , total integrated cost =  105.36750189638758
RUN  4 , total integrated cost =  70.83543334316654
RUN  5 , total integrated cost =  48.11700954954317
RUN  6 , total integrated cost =  33.12636673097042
RUN  7 , total integrated cost =  32.7980859536464
RUN  8 , total integrated cost =  32.292902984232086
RUN  9 , total integrated cost =  31.96993065365211
RUN  10 , tot

...

RUN  5 , total integrated cost =  12332.83832147594
RUN  6 , total integrated cost =  12326.923571729154
RUN  7 , total integrated cost =  12326.92357172915
RUN  8 , total integrated cost =  12326.923571729149
RUN  9 , total integrated cost =  12326.923571729149
Control only changes marginally.
RUN  9 , total integrated cost =  12326.923571729149
Improved over  9  iterations in  0.3206965085119009  seconds by  4.632644777953217  percent.
Problem in initial value trasfer:  Vmean_exc -56.67029961519106 -56.67030847749664
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8231.907221468136
Gradient descend method:  None
RUN  1 , total integrated cost =  8231.907221468136
Control only changes marginally.
RUN  1 , total integrated cost =  8231.907221468136
Improved over  1  iterations in  0.05727574788033962  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolat

ERROR:root:Problem in initial value trasfer


RUN  80 , total integrated cost =  53.8583729602392
Control only changes marginally.
RUN  81 , total integrated cost =  53.8583729602392
Improved over  81  iterations in  1.691040989011526  seconds by  99.84386989805198  percent.
Problem in initial value trasfer:  Vmean_exc -56.70310167417908 -56.703102732259566
weight =  6404.914795565372
set cost params:  1.0 0.0 6404.914795565372
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  34080.169259303424
Gradient descend method:  None
RUN  1 , total integrated cost =  31765.061415100372
RUN  2 , total integrated cost =  31764.407794817405
RUN  3 , total integrated cost =  31760.31709944645
RUN  4 , total integrated cost =  31698.063452564045
RUN  5 , total integrated cost =  31681.604674489125
RUN  6 , total integrated cost =  31681.029446312747
RUN  7 , total integrated cost =  31680.93942085337
RUN  8 , total integrated cost =  31680.8933646668
RUN  9 , total integrated cost =  31680.850533753626
RUN  10 , total int

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  31478.261892334984
Improved over  26  iterations in  0.5951921474188566  seconds by  7.634666797490013  percent.
Problem in initial value trasfer:  Vmean_exc -56.70311918212211 -56.70311920300496
-------  85 0.47500000000000014 0.7250000000000004
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15143.755110304457
Gradient descend method:  None
RUN  1 , total integrated cost =  15143.755110304457
Control only changes marginally.
RUN  1 , total integrated cost =  15143.755110304457
Improved over  1  iterations in  0.05082132667303085  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15143.755110304457
Gradient descend method:  None
RUN  1 , total integrated cost =  15143.755110304457
Control only changes marginally.
RUN  1 , total integrated cost =  15143.755110304457
Improv

In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  52.85822227638283
Control only changes marginally.
RUN  60 , total integrated cost =  52.85822227638283
Improved over  60  iterations in  1.304267404600978  seconds by  99.36063550672078  percent.
Problem in initial value trasfer:  Vmean_exc -56.63980652406424 -56.63980643091702
weight =  1557.3560492491572
set cost params:  1.0 0.0 1557.3560492491572
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8207.164683268185
Gradient descend method:  None
RUN  1 , total integrated cost =  7979.887213839216
RUN  2 , total integrated cost =  7979.782248912283
RUN  3 , total integrated cost =  7979.72825896998
RUN  4 , total integrated cost =  7979.588665197086
RUN  5 , total integrated cost =  7932.731209219346
RUN  6 , total integrated cost =  7932.614544037599
RUN  7 , total integrated cost =  7932.614526884132


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  7932.61452674689
RUN  9 , total integrated cost =  7932.614526746067
RUN  10 , total integrated cost =  7932.614526746065
RUN  11 , total integrated cost =  7932.614526746063
RUN  12 , total integrated cost =  7932.614526746063
Control only changes marginally.
RUN  12 , total integrated cost =  7932.614526746063
Improved over  12  iterations in  0.3121395409107208  seconds by  3.3452497557633194  percent.
Problem in initial value trasfer:  Vmean_exc -56.638178991269825 -56.638201701727034
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
[0, 5, 10, 15, 20, 30, 35, 40] []
closest index  40
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20662.11093044098
Gradient

...

RUN  17 , total integrated cost =  126.44573423200414
RUN  18 , total integrated cost =  126.44573422849282
RUN  19 , total integrated cost =  126.44573422845401
RUN  20 , total integrated cost =  126.44573422845343
Control only changes marginally.
RUN  25 , total integrated cost =  126.4457342284533
Improved over  25  iterations in  0.5803964044898748  seconds by  99.13083642854042  percent.
Problem in initial value trasfer:  Vmean_exc -56.67729530929731 -56.67729527816752
-------  130 0.6000000000000003 0.8500000000000005
found solution for  130
-------  135 0.5250000000000001 0.8750000000000006
[0, 5, 10, 15, 20, 30, 35, 40, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130] []
closest index  120
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23620.15202322861
Gradient descend method:  None
RUN  1 , total integrated cost =  23532.68177719166
RUN  2 , total integrated cost =  23532.63628042645
RUN  3 , to

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  126.2173952654266
RUN  9 , total integrated cost =  126.2173952654266
Control only changes marginally.
RUN  9 , total integrated cost =  126.2173952654266
Improved over  9  iterations in  0.23792347125709057  seconds by  99.46364956948324  percent.
Problem in initial value trasfer:  Vmean_exc -56.70082022669904 -56.70081312880305
-------  140 0.4500000000000001 0.9000000000000006
found solution for  140
-------  145 0.5750000000000002 0.9000000000000006
[0, 5, 10, 15, 20, 30, 35, 40, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130, 140] []
closest index  130
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  298.41704277226154
Gradient descend method:  None
RUN  1 , total integrated cost =  298.41704277226154
Control only changes marginally.
RUN  1 , total integrated cost =  298.41704277226154
Improved over  1  iterations in  0.08846349082887173  seconds by  0.0  percent.
Pro

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  27814.768115681818
Control only changes marginally.
RUN  1 , total integrated cost =  27814.768115681818
Improved over  1  iterations in  0.08920356817543507  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70333201668662 -56.70334396076825
------------------------------------------------------------
-------------------- 1
------------------------------------------------------------
found solution:  [0, 5, 10, 15, 20, 30, 35, 40, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130, 140]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
found solution for  25
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2026.6618762892883
set cost params:  1.0 0.0 2026.6618762892883
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5091.424407996941
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5091.424407996941
Control only changes marginally.
RUN  1 , total integrated cost =  5091.424407996941
Improved over  1  iterations in  0.07040444202721119  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62659014344855 -56.62656380958566
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  4939.573061485599
set cost params:  1.0 0.0 4939.573061485599
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13010.206794263859
Gradient descend method:  None
RUN  1 , total integrated cost =  13010.098778658572
RUN  2 , total integrated cost =  13010.09846774978


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  13010.098467749774
RUN  4 , total integrated cost =  13010.098467749774
Control only changes marginally.
RUN  4 , total integrated cost =  13010.098467749774
Improved over  4  iterations in  0.16896388493478298  seconds by  0.0008326271503449334  percent.
Problem in initial value trasfer:  Vmean_exc -56.670288159146004 -56.67029729015339
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1615.1141405505755
set cost params:  1.0 0.0 1615.1141405505755
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8224.103115670541
Gradient descend method:  None
RUN  1 , total integrated cost =  8224.0613902543
RUN  2 , total integrated cost =  8224.061375568508
RUN  3 , total integrated cost =  8224.06137555303
RUN  4 , total integrated cost =  8224.061375552892


...

RUN  1 , total integrated cost =  14535.36652589004
Control only changes marginally.
RUN  1 , total integrated cost =  14535.36652589004
Improved over  1  iterations in  0.07608734630048275  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67723734484127 -56.67723871735414
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  135 0.5250000000000001 0.8750000000000006
no convergence
weight =  2156.2180763759725
set cost params:  1.0 0.0 2156.2180763759725
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23521.727023739644
Gradient descend method:  None
RUN  1 , total integrated cost =  23521.727023736243
RUN  2 , total integrated cost =  23521.727023735813
RUN  3 , total integrated cost =  23521.727023735755
RUN  4 , total integrated cost =  23521.727023735748


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  23521.727023735748
Control only changes marginally.
RUN  5 , total integrated cost =  23521.727023735748
Improved over  5  iterations in  0.215575048699975  seconds by  1.6569856597925536e-11  percent.
Problem in initial value trasfer:  Vmean_exc -56.70068481853917 -56.7006843934112
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
[[True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Problem in initial value trasfer:  Vmean_exc -56.63814278851488 -56.63816600111047
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  95 0.5250000000000001 

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  125.19210669416819
Gradient descend method:  None
RUN  1 , total integrated cost =  26.01179625955184
RUN  2 , total integrated cost =  25.986236372847284
RUN  3 , total integrated cost =  25.986153288375004
RUN  4 , total integrated cost =  25.986152983218687
RUN  5 , total integrated cost =  25.986152981213486
RUN  6 , total integrated cost =  25.98615298118533
RUN  7 , total integrated cost =  25.98615298118519


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  25.98615298118519
Control only changes marginally.
RUN  8 , total integrated cost =  25.98615298118519
Improved over  8  iterations in  0.748466856777668  seconds by  79.24297811788824  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447040208967 -56.624470247888716
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  259.6535601139009
Gradient descend method:  HS
RUN  1 , total integrated cost =  258.85850323280044
RUN  2 , total integrated cost =  258.4433677360224
RUN  3 , total integrated cost =  258.44336773602225
RUN  4 , total integrated cost =  258.4433677360222


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  258.4433677360222
Control only changes marginally.
RUN  5 , total integrated cost =  258.4433677360222
Improved over  5  iterations in  0.6693605687469244  seconds by  0.4660796398662228  percent.
Problem in initial value trasfer:  Vmean_exc -56.624468158958244 -56.624467403568914
weight =  1971.3043670465433
set cost params:  1.0 0.0 1971.3043670465433
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5076.4441709180865
Gradient descend method:  None
RUN  1 , total integrated cost =  4949.07791121275
RUN  2 , total integrated cost =  4949.076613620341
RUN  3 , total integrated cost =  4949.076613620338
RUN  4 , total integrated cost =  4949.076613620336


...

RUN  1 , total integrated cost =  298.41704277226154
Control only changes marginally.
RUN  1 , total integrated cost =  298.41704277226154
Improved over  1  iterations in  0.25464347191154957  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70333201668662 -56.70334396076825
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2538.4434714584463
Gradient descend method:  HS


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2538.4434714584463
Control only changes marginally.
RUN  1 , total integrated cost =  2538.4434714584463
Improved over  1  iterations in  0.2532423064112663  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70333201668662 -56.70334396076825
weight =  1310.4356037934983
set cost params:  1.0 0.0 1310.4356037934983
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32665.19616834907
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32665.19616834907
Control only changes marginally.
RUN  1 , total integrated cost =  32665.19616834907
Improved over  1  iterations in  0.2542037330567837  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70333201668662 -56.70334396076825


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2029.3403004063673
set cost params:  1.0 0.0 2029.3403004063673
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.226626394167
Gradient descend method:  None
RUN  1 , total integrated cost =  5092.181822074602
RUN  2 , total integrated cost =  5092.181810945805
RUN  3 , total integrated cost =  5092.1818109397245
RUN  4 , total integrated cost =  5092.181810939709
RUN  5 , total integrated cost =  5092.181810939708


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5092.181810939708
Control only changes marginally.
RUN  6 , total integrated cost =  5092.181810939708
Improved over  6  iterations in  0.8640426397323608  seconds by  0.0008800758047016188  percent.
Problem in initial value trasfer:  Vmean_exc -56.626216915281894 -56.62619253038245
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  4940.6751383372375
set cost params:  1.0 0.0 4940.6751383372375
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13012.126087695231
Gradient descend method:  None
RUN  1 , total integrated cost =  13012.100180366013


ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  13012.100180366007
RUN  3 , total integrated cost =  13012.100180366007
Control only changes marginally.
RUN  3 , total integrated cost =  13012.100180366007
Improved over  3  iterations in  0.5148761477321386  seconds by  0.0001991014308373451  percent.
Problem in initial value trasfer:  Vmean_exc -56.670289510783036 -56.670298599719516
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1619.1656933315635
set cost params:  1.0 0.0 1619.1656933315635
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8225.187226970553
Gradient descend method:  None
RUN  1 , total integrated cost =  8225.171021651193
RUN  2 , total integrated cost =  8225.169664686717
RUN  3 , total integrated cost =  8225.169444453099
RUN  4 , total integrated cost =  8225.169291836495
RUN  5 , total integrated cost =  8225.169222759869
RUN  6 , total integrated cost =  8225.169160052945
RUN  7 , total integrated cost =  8225.169127937223
RU

...

RUN  4 , total integrated cost =  20622.793620031465
RUN  5 , total integrated cost =  20622.793620031465
Control only changes marginally.
RUN  5 , total integrated cost =  20622.793620031465
Improved over  5  iterations in  0.8072860073298216  seconds by  5.4143356464919634e-12  percent.
Problem in initial value trasfer:  Vmean_exc -56.69638510564511 -56.69638632207293
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  75 0.5750000000000002 0.6750000000000004
no convergence
weight =  22551.34721735198
set cost params:  1.0 0.0 22551.34721735198
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  34494.29933183394
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  34494.29933183394
Control only changes marginally.
RUN  1 , total integrated cost =  34494.29933183394
Improved over  1  iterations in  0.2046978622674942  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70311113152748 -56.703111980884
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 3
[[True, True], [False, False], [True, True], [True, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, False], [Tru

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  20622.794273896016
Control only changes marginally.
RUN  1 , total integrated cost =  20622.794273896016
Improved over  1  iterations in  0.20119465701282024  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69638510564511 -56.69638632207293
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 4
[[True, True], [True, False], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True

In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  92.48392920448008
Gradient descend method:  None
RUN  1 , total integrated cost =  54.68298316294973
RUN  2 , total integrated cost =  11.724745082215428
RUN  3 , total integrated cost =  7.942083804675207
RUN  4 , total integrated cost =  6.92468466384047
RUN  5 , total integrated cost =  6.452103451978884
RUN  6 , total integrated cost =  6.140113330032701
RUN  7 , total integrated cost =  5.921786851859703
RUN  8 , total integrated cost =  5.742124517616148
RUN  9 , total integrated cost =  5.596766067254773
RUN  10 , total integrated cost =  5.463650101195615
RUN  11 , total integrated cost =  5.348952238247388
RUN  12 , total integrated cost =  5.228326032773699
RUN  13 , total integrated cost =  5.123119762256654
RUN  14 , total integrated cost =  5.002478524151401
RUN  15 , total integrated cost =  4.894502008996814
RUN  16 , tot

ERROR:root:Problem in initial value trasfer


RUN  70 , total integrated cost =  5.27973482016793
Control only changes marginally.
RUN  76 , total integrated cost =  5.2797348201675325
Improved over  76  iterations in  1.570002906024456  seconds by  93.9483724776236  percent.
Problem in initial value trasfer:  Vmean_exc -56.639804819477305 -56.63980478030708
-------  45 0.5000000000000002 0.5750000000000003
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  221.16539614071414
Gradient descend method:  None
RUN  1 , total integrated cost =  5.4538917952034875
RUN  2 , total integrated cost =  5.452425267391442
RUN  3 , total integrated cost =  5.430126535024788
RUN  4 , total integrated cost =  5.424230389506987
RUN  5 , total integrated cost =  5.424013597671528
RUN  6 , total integrated cost =  5.423681532653847
RUN  7 , total integrated cost =  5.420716507474317
RUN  8 , total integrated cost =  5.419893125584101
RUN  9 , total integrated cost =  5.419814601310721
RUN  10 , tota

ERROR:root:Problem in initial value trasfer


RUN  50 , total integrated cost =  5.372948535619039
Control only changes marginally.
RUN  57 , total integrated cost =  5.3729485356190105
Improved over  57  iterations in  1.214194756001234  seconds by  97.57061971294979  percent.
Problem in initial value trasfer:  Vmean_exc -56.696426603891275 -56.69642656985887
-------  65 0.5000000000000002 0.6500000000000004
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  262.18119570475926
Gradient descend method:  None
RUN  1 , total integrated cost =  7.870516620077838
RUN  2 , total integrated cost =  7.862535455395315
RUN  3 , total integrated cost =  7.781754012451625
RUN  4 , total integrated cost =  7.770882901040214
RUN  5 , total integrated cost =  7.769536260446904
RUN  6 , total integrated cost =  7.7629315739275375
RUN  7 , total integrated cost =  7.760584482581817
RUN  8 , total integrated cost =  7.759721708255701
RUN  9 , total integrated cost =  7.754111681996597
RUN  10 , to

...

RUN  5 , total integrated cost =  12.641005209953079
RUN  6 , total integrated cost =  12.641005209940007
RUN  7 , total integrated cost =  12.641005209939541
RUN  8 , total integrated cost =  12.641005209939525
RUN  9 , total integrated cost =  12.641005209939522
RUN  10 , total integrated cost =  12.64100520993952
RUN  11 , total integrated cost =  12.641005209939516
RUN  12 , total integrated cost =  12.641005209939516
Control only changes marginally.
RUN  12 , total integrated cost =  12.641005209939516
Improved over  12  iterations in  0.28211300261318684  seconds by  32.2294471594231  percent.
Problem in initial value trasfer:  Vmean_exc -56.677305891478696 -56.67730556589831
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17.38719689657785
Gradient descend method:  None
RUN  1 , total integrated cost =  10.903901536721774
RUN  2 , total integrated cost =  10.903630378351314
R

ERROR:root:Problem in initial value trasfer


RUN  11 , total integrated cost =  10.903624188512
RUN  12 , total integrated cost =  10.903624188512
Control only changes marginally.
RUN  12 , total integrated cost =  10.903624188512
Improved over  12  iterations in  0.2896233331412077  seconds by  37.28935001214572  percent.
Problem in initial value trasfer:  Vmean_exc -56.70068319668632 -56.70068289947422
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  74.41439990364303
Gradient descend method:  None
RUN  1 , total integrated cost =  43.38837482064854
RUN  2 , total integrated cost =  38.65839499836778
RUN  3 , total integrated cost =  28.32793499478849
RUN  4 , total integrated cost =  25.22867393726211
RUN  5 , total integrated cost =  22.572311107194373
RUN  6 , total integrated cost =  17.189162027752264
RUN  7 , total integrated cost =  17.019469754226908
RUN  8 , total integrated cost =  16.813248809845675
RUN  9 , total 

ERROR:root:Problem in initial value trasfer


RUN  100 , total integrated cost =  9.215515203822703
Control only changes marginally.
RUN  107 , total integrated cost =  9.215515174761341
Improved over  107  iterations in  2.2039080392569304  seconds by  87.61595176915458  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354256628889 -56.70354250943042


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.5969409253435503
Gradient descend method:  None
RUN  1 , total integrated cost =  2.5969409253435503
Control only changes marginally.
RUN  1 , total integrated cost =  2.5969409253435503
Improved over  1  iterations in  0.06179500371217728  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002
no convergence
set cost params:  1.0 0.0 1.0
interp

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5.2797348201675325
Control only changes marginally.
RUN  1 , total integrated cost =  5.2797348201675325
Improved over  1  iterations in  0.061716899275779724  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.639804819477305 -56.63980478030708
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.3729485356190105
Gradient descend method:  None
RUN  1 , total integrated cost =  5.3729485356190105
Control only changes marginally.
RUN  1 , total integrated cost =  5.3729485356190105
Improved over  1  iterations in  0.06235392391681671  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.696426603891275 -56.69642656985887
-------  65 0.5000000000000002 0.6500000000000004
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.6934082

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.7635084796625167
Control only changes marginally.
RUN  1 , total integrated cost =  1.7635084796625167
Improved over  1  iterations in  0.06282682903110981  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70311037591106 -56.703111247160486
-------  85 0.47500000000000014 0.7250000000000004
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9.623736158983492
Gradient descend method:  None
RUN  1 , total integrated cost =  9.623736158983492
Control only changes marginally.
RUN  1 , total integrated cost =  9.623736158983492
Improved over  1  iterations in  0.06212068907916546  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67996049484737 -56.67996040172404
-------  95 0.5250000000000001 0.7500000000000004
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.22873167082

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  14.592961043624294
Control only changes marginally.
RUN  1 , total integrated cost =  14.592961043624294
Improved over  1  iterations in  0.06501591764390469  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6244092530479 -56.62440559377636
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12.641005209939516
Gradient descend method:  None
RUN  1 , total integrated cost =  12.641005209939516
Control only changes marginally.
RUN  1 , total integrated cost =  12.641005209939516
Improved over  1  iterations in  0.06259383447468281  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.677305891478696 -56.67730556589831
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10.903624

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  9.215515174761341
Control only changes marginally.
RUN  1 , total integrated cost =  9.215515174761341
Improved over  1  iterations in  0.062444962561130524  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354256628889 -56.70354250943042
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
